# Step 3: Feature Engineering

**Owner:** P4  
**Model:** Feature-Based MLP  
**Dataset:** MIND-large


In [1]:

# ==========================================================
# 1. GOOGLE COLAB SETUP
# ==========================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import sys
import subprocess
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# Locate or clone GitHub repository
PROJECT_ROOT = Path("/content/BT4222-Project-Group-4--MIND")

if not (PROJECT_ROOT / "config.py").exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/Fdiannz/BT4222-Project-Group-4--MIND.git",
        str(PROJECT_ROOT)
    ], check=True)

sys.path.insert(0, str(PROJECT_ROOT))

# Correct shared Google Drive folder
DATA_ROOT = Path(
    "/content/drive/MyDrive/BT4222 G4/BT4222_Project"
)

os.environ["MIND_PROJECT_DIR"] = str(DATA_ROOT)

import config
importlib.reload(config)

PROCESSED_DIR = Path(config.PROCESSED_DIR)

# Validate paths
assert (PROJECT_ROOT / "config.py").exists()
assert DATA_ROOT.is_dir(), "Project folder not found"
assert PROCESSED_DIR.is_dir(), "Processed folder not found"

print("Repository found:", True)
print("Project folder found:", DATA_ROOT.is_dir())
print("Processed folder found:", PROCESSED_DIR.is_dir())
print("Setup successful!")

Mounted at /content/drive
Repository found: True
Project folder found: True
Processed folder found: True
Setup successful!


In [ ]:
# ==========================================================
# 2. INSPECT PROCESSED DATA
# ==========================================================

required_files = [
    "news_metadata.parquet",
    "fit_impressions.parquet",
    "tune_impressions.parquet",
    "eval_impressions.parquet",
    "fit_candidates_sampled.parquet",
    "tune_candidates.parquet",
    "eval_candidates.parquet",
]

for filename in required_files:
    path = PROCESSED_DIR / filename

    if not path.exists():
        print(f"MISSING: {filename}")
        continue

    pf = pq.ParquetFile(path)

    print(f"\n=== {filename} ===")
    print(f"Rows: {pf.metadata.num_rows:,}")
    print("Columns:", pf.schema_arrow.names)

# Inspect impression history structure
for split in ["fit", "tune", "eval"]:
    path = PROCESSED_DIR / f"{split}_impressions.parquet"

    if not path.exists():
        continue

    pf = pq.ParquetFile(path)

    sample = next(
        pf.iter_batches(batch_size=2)
    ).to_pandas()

    print(f"\n=== {split.upper()} SAMPLE ===")
    display(sample)


=== news_metadata.parquet ===
Rows: 104,151
Columns: ['news_id', 'category', 'subcategory', 'title', 'abstract', 'url', 'title_entities', 'abstract_entities', 'news_idx']

=== fit_impressions.parquet ===
Rows: 1,801,231
Columns: ['impression_id', 'user_id', 'time', 'history', 'impressions', 'user_idx']

=== tune_impressions.parquet ===
Rows: 431,517
Columns: ['impression_id', 'user_id', 'time', 'history', 'impressions', 'user_idx']

=== eval_impressions.parquet ===
Rows: 376,471
Columns: ['impression_id', 'user_id', 'time', 'history', 'impressions', 'user_idx']

=== fit_candidates_sampled.parquet ===
Rows: 12,894,210
Columns: ['impression_id', 'user_id', 'user_idx', 'time', 'news_id', 'news_idx', 'label', 'prior_exposure_count', 'cold_article', 'category', 'subcategory']

=== tune_candidates.parquet ===
Rows: 17,400,106
Columns: ['impression_id', 'user_id', 'user_idx', 'time', 'news_id', 'news_idx', 'label', 'prior_exposure_count', 'cold_article', 'category', 'subcategory']

=== eval_

,impression_id,user_id,time,history,impressions,user_idx
0,1,U87243,2019-11-10 11:30:54,N8668 N39081 N65259 N79529 N73408 N43615 N2937...,N78206-0 N26368-0 N7578-0 N58592-0 N19858-0 N5...,1
1,2,U598644,2019-11-12 13:45:29,N56056 N8726 N70353 N67998 N83823 N111108 N107...,N47996-0 N82719-0 N117066-0 N8491-0 N123784-0 ...,2



=== TUNE SAMPLE ===


,impression_id,user_id,time,history,impressions,user_idx
0,5,U239687,2019-11-14 20:03:01,N65250 N122359 N71723 N53796 N41663 N41484 N11...,N76209-0 N48841-0 N67937-0 N62235-0 N6307-0 N3...,5
1,11,U108656,2019-11-14 06:51:18,N4833 N61319 N94639 N50163 N107002 N112041 N49...,N95301-0 N79081-0 N103133-0 N80281-0 N70363-0 ...,11



=== EVAL SAMPLE ===


,impression_id,user_id,time,history,impressions,user_idx
0,1,U134050,2019-11-15 08:55:22,N12246 N128820 N119226 N4065 N67770 N33446 N10...,N91737-0 N30206-0 N54368-0 N117802-0 N18190-0 ...,89712
1,2,U254959,2019-11-15 11:42:35,N34011 N9375 N67397 N7936 N118985 N109453 N103...,N119999-0 N24958-0 N104054-0 N33901-0 N9250-0 ...,516861


In [ ]:
# ==========================================================
# 3. CATEGORY AND SUBCATEGORY MAPPINGS
# ==========================================================

import json

# Load article metadata
news_metadata = pd.read_parquet(
    PROCESSED_DIR / "news_metadata.parquet",
    columns=["news_id", "category", "subcategory"]
)

# Read FIT candidates in batches to limit memory usage
fit_file = pq.ParquetFile(
    PROCESSED_DIR / "fit_candidates_sampled.parquet"
)

fit_categories = set()
fit_subcategories = set()

for batch in fit_file.iter_batches(
    batch_size=100_000,
    columns=["category", "subcategory"]
):
    df = batch.to_pandas()

    fit_categories.update(
        df["category"].dropna().astype(str).unique()
    )

    fit_subcategories.update(
        df["subcategory"].dropna().astype(str).unique()
    )

# Create fit-only categorical vocabularies
# Reserve index 0 for unseen or missing categories
def build_vocab(values):
    clean = sorted(
        str(x) for x in values
        if str(x).strip()
    )
    return {value: i + 1 for i, value in enumerate(clean)}

category_to_idx = build_vocab(fit_categories)
subcategory_to_idx = build_vocab(fit_subcategories)

# Save the mappings so the MLP uses identical encodings
vocab_path = PROCESSED_DIR / "step3_category_vocab.json"

with open(vocab_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "category_to_idx": category_to_idx,
            "subcategory_to_idx": subcategory_to_idx
        },
        f,
        indent=2
    )

print("News metadata:", len(news_metadata))
print("FIT categories:", len(category_to_idx))
print("FIT subcategories:", len(subcategory_to_idx))
print("Vocabulary saved:", vocab_path)

display(news_metadata.head())

News metadata: 104151
FIT categories: 16
FIT subcategories: 218
Vocabulary saved: G:\.shortcut-targets-by-id\1D4Db1s7DJ5PVK9CFiV-pklpCW5dr1uOM\BT4222_Project\data\processed\step3_category_vocab.json


,news_id,category,subcategory
0,N88753,lifestyle,lifestyleroyals
1,N45436,news,newsscienceandtechnology
2,N23144,health,weightloss
3,N86255,health,medical
4,N93187,news,newsworld


In [ ]:
# ==========================================================
# 4. USER-HISTORY FEATURE FUNCTIONS
# ==========================================================
#these are the personalised features

# Lookup: article ID -> article category
news_to_category = dict(
    zip(
        news_metadata["news_id"],
        news_metadata["category"]
    )
)

# Convert space-separated history into article IDs
def parse_history(history):
    if not isinstance(history, str) or not history.strip():
        return []
    return history.split()

# Feature 1: Number of previously clicked articles
def get_hist_len(history):
    return len(parse_history(history))

# Feature 2: Share of history matching candidate category
def get_cat_match(history, candidate_news_id):
    history_ids = parse_history(history)

    if not history_ids:
        return 0.0

    candidate_category = news_to_category.get(
        candidate_news_id
    )

    if pd.isna(candidate_category):
        return 0.0

    matches = sum(
        news_to_category.get(news_id) == candidate_category
        for news_id in history_ids
    )

    return matches / len(history_ids)

print("User-history feature functions ready.")

User-history feature functions ready.


In [ ]:
# ==========================================================
# 5. VALIDATE USER-HISTORY FEATURES
# ==========================================================
#testing the ffunctions on a small sample of FIT impressions

fit_impressions_file = pq.ParquetFile(
    PROCESSED_DIR / "fit_impressions.parquet"
)

sample = next(
    fit_impressions_file.iter_batches(
        batch_size=10,
        columns=["impression_id", "history", "impressions"]
    )
).to_pandas()

# Get the first displayed candidate from each impression
def first_candidate(impressions):
    if not isinstance(impressions, str) or not impressions.strip():
        return None

    first = impressions.split()[0]

    # Format: news_id-click_label
    return first.rsplit("-", 1)[0]


sample["candidate_news_id"] = (
    sample["impressions"].apply(first_candidate)
)

sample["hist_len"] = (
    sample["history"].apply(get_hist_len)
)

sample["cat_match"] = sample.apply(
    lambda row: get_cat_match(
        row["history"],
        row["candidate_news_id"]
    ),
    axis=1
)

display(
    sample[
        [
            "impression_id",
            "candidate_news_id",
            "hist_len",
            "cat_match"
        ]
    ]
)

# Basic validation
assert (sample["hist_len"] >= 0).all()
assert sample["cat_match"].between(0, 1).all()

print("History feature validation passed!")

FIT rows: 12894210
FIT row groups: 133


In [ ]:
#%pip install sentence-transformers

   ---------------------------------------- 0.0/760.7 kB ? eta -:--:--
   --------------------------------------- 760.7/760.7 kB 16.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/857.0 kB ? eta -:--:--
   --------------------------------------- 857.0/857.0 kB 19.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ---------------------------------------- 2.9/2.9 MB 18.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/13.1 MB ? eta -:--:--
   ------------ --------------------------- 4.2/13.1 MB 25.2 MB/s eta 0:00:01
   ---------------------------------------  12.8/13.1 MB 33.6 MB/s eta 0:00:01
   ---------------------------------------- 13.1/13.1 MB 30.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/3.9 MB ? eta -:--:--
   ---------------------------------------- 3.9/3.9 MB 29.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# ==========================================================
# 6. ARTICLE TEXT EMBEDDINGS - SETUP
# ==========================================================

from pathlib import Path
import torch
from sentence_transformers import SentenceTransformer

# Store embeddings separately from processed features
EMBEDDINGS_DIR = PROCESSED_DIR.parent / "embeddings"
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

# Use GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

# Pretrained model for semantic text embeddings
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

print("Embedding model loaded successfully.")
print("Embedding dimensions:",
      embedding_model.get_sentence_embedding_dimension())


c:\Users\eishw\OneDrive - National University of Singapore\BT4222\Project\BT4222-Project-Group-4--MIND\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


c:\Users\eishw\OneDrive - National University of Singapore\BT4222\Project\BT4222-Project-Group-4--MIND\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\eishw\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\eishw\OneDri

Embedding model loaded successfully.
Embedding dimensions: 384


C:\Users\eishw\AppData\Local\Temp\ipykernel_28508\1286576246.py:26: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_model.get_sentence_embedding_dimension())


In [ ]:

# ==========================================================
# 7. PREPARE ARTICLE TEXT
# ==========================================================

news_text = pd.read_parquet(
    PROCESSED_DIR / "news_metadata.parquet",
    columns=["news_id", "title", "abstract"]
)

# Ensure one embedding per unique article
news_text = (
    news_text
    .dropna(subset=["news_id"])
    .drop_duplicates(subset=["news_id"])
    .sort_values("news_id")
    .reset_index(drop=True)
)

# Handle missing text safely
news_text["title"] = (
    news_text["title"].fillna("").astype(str)
)

news_text["abstract"] = (
    news_text["abstract"].fillna("").astype(str)
)

news_text["article_text"] = (
    news_text["title"].str.strip()
    + ". "
    + news_text["abstract"].str.strip()
).str.strip()

print("Total unique articles:", len(news_text))
print("Missing/empty text:",
      (news_text["article_text"] == "").sum())

display(
    news_text[
        ["news_id", "title", "abstract"]
    ].head()
)

Total unique articles: 104151
Missing/empty text: 0


,news_id,title,abstract
0,N1,Texans defensive tackle D.J. Reader is taking ...,Houston Texans defensive tackle D.J. Reader is...
1,N100,Fed Seen Cutting Rates Next Week and Then Hitt...,Federal Reserve officials will signal they're ...
2,N1000,Severe Weather Leaves Thousands Without Power,There were tense moments overnight as the Nati...
3,N10000,Stars attend the 2019 Governors Awards for the...,See which celebs attended the 2019 Governors A...
4,N100000,Ahkello Witherspoon likely to return Week 11 v...,Witherspoon practiced Friday


In [ ]:

# ==========================================================
# 8. TEST ARTICLE EMBEDDINGS
# ==========================================================

test_texts = news_text["article_text"].head(5).tolist()

test_embeddings = embedding_model.encode(
    test_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Test embedding shape:", test_embeddings.shape)

assert test_embeddings.shape == (5, 384)
assert np.isfinite(test_embeddings).all()

print("Article embedding test passed!")

Test embedding shape: (5, 384)
Article embedding test passed!


In [ ]:
# ==========================================================
# 9. GENERATE AND SAVE ALL ARTICLE EMBEDDINGS
# ==========================================================

from pathlib import Path

embeddings_path = EMBEDDINGS_DIR / "news_embeddings.npy"
ids_path = EMBEDDINGS_DIR / "news_embedding_ids.parquet"

# Avoid repeating the expensive encoding process
if embeddings_path.exists() and ids_path.exists():
    print("Existing embedding files found. Validating...")

    news_embeddings = np.load(
        embeddings_path, mmap_mode="r"
    )
    saved_ids = pd.read_parquet(ids_path)

    assert news_embeddings.shape == (len(news_text), 384)
    assert saved_ids["news_id"].tolist() == (
        news_text["news_id"].tolist()
    )

    print("Existing embeddings are valid. Skipping generation.")

else:
    if embeddings_path.exists() != ids_path.exists():
        raise FileNotFoundError(
            "Only one embedding file exists. "
            "Check the embedding folder before rerunning."
        )

    print(f"Generating embeddings for {len(news_text):,} articles...")

    # Smaller batches for CPU; larger batches for GPU
    batch_size = 128 if device == "cuda" else 32

    news_embeddings = embedding_model.encode(
        news_text["article_text"].tolist(),
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    ).astype(np.float32)

    # Validate before saving
    assert news_embeddings.shape == (len(news_text), 384)
    assert np.isfinite(news_embeddings).all()

    # Save embeddings
    np.save(embeddings_path, news_embeddings)

    # Save article IDs in the exact same row order
    pd.DataFrame({
        "news_id": news_text["news_id"]
    }).to_parquet(ids_path, index=False)

    print("All embeddings generated and saved!")

print("\n=== FINAL EMBEDDING CHECK ===")
print("Embedding shape:", news_embeddings.shape)
print("Embedding file:", embeddings_path)
print("Article ID file:", ids_path)
print("Number of article IDs:", len(news_text))


Generating embeddings for 104,151 articles...


Batches: 100%|██████████| 3255/3255 [1:04:10<00:00,  1.18s/it]  


All embeddings generated and saved!

=== FINAL EMBEDDING CHECK ===
Embedding shape: (104151, 384)
Embedding file: G:\.shortcut-targets-by-id\1D4Db1s7DJ5PVK9CFiV-pklpCW5dr1uOM\BT4222_Project\data\embeddings\news_embeddings.npy
Article ID file: G:\.shortcut-targets-by-id\1D4Db1s7DJ5PVK9CFiV-pklpCW5dr1uOM\BT4222_Project\data\embeddings\news_embedding_ids.parquet
Number of article IDs: 104151


In [ ]:

# ==========================================================
# 10. LOAD EMBEDDINGS AND CREATE ARTICLE LOOKUP
# ==========================================================

# Load the saved article embeddings
news_embeddings = np.load(
    EMBEDDINGS_DIR / "news_embeddings.npy",
    mmap_mode="r"
)

embedding_ids = pd.read_parquet(
    EMBEDDINGS_DIR / "news_embedding_ids.parquet"
)

# Each article ID points to its corresponding embedding row
news_to_embedding_idx = {
    str(news_id): idx
    for idx, news_id in enumerate(embedding_ids["news_id"])
}

# Validation
assert len(embedding_ids) == len(news_embeddings)
assert embedding_ids["news_id"].is_unique

print("Embeddings loaded:", news_embeddings.shape)
print("Article IDs mapped:", len(news_to_embedding_idx))
print("Article lookup ready!")

Embeddings loaded: (104151, 384)
Article IDs mapped: 104151
Article lookup ready!


In [ ]:
# ==========================================================
# 11. USER-ARTICLE SEMANTIC SIMILARITY
# ==========================================================

def get_user_profile(history):
    """
    Represent a user's interests using the mean embedding
    of articles clicked before the current impression.
    """

    history_ids = parse_history(history)

    # Find articles with available embeddings
    indices = [
        news_to_embedding_idx[nid]
        for nid in history_ids
        if nid in news_to_embedding_idx
    ]

    if not indices:
        return None

    # Average previously clicked article embeddings
    profile = np.asarray(
        news_embeddings[indices],
        dtype=np.float32
    ).mean(axis=0)

    # Normalise for cosine similarity
    norm = np.linalg.norm(profile)

    if norm == 0:
        return None

    return profile / norm


def get_user_article_sim(history, candidate_news_id):
    """
    Calculate cosine similarity between the user's
    reading history and a candidate article.
    """

    user_profile = get_user_profile(history)

    candidate_idx = news_to_embedding_idx.get(
        str(candidate_news_id)
    )

    # Missing history or missing candidate embedding
    if user_profile is None or candidate_idx is None:
        return 0.0

    candidate_embedding = news_embeddings[candidate_idx]

    similarity = np.dot(
        user_profile,
        candidate_embedding
    )

    return float(np.clip(similarity, -1.0, 1.0))


print("User-article similarity functions ready!")


User-article similarity functions ready!


In [ ]:

# ==========================================================
# 12. VALIDATE USER-ARTICLE SIMILARITY
# ==========================================================

# Load a small sample directly from P1's FIT impressions
fit_impressions_file = pq.ParquetFile(
    PROCESSED_DIR / "fit_impressions.parquet"
)

sample = next(
    fit_impressions_file.iter_batches(
        batch_size=10,
        columns=["impression_id", "history", "impressions"]
    )
).to_pandas()

# Extract the first candidate news ID from each impression
def first_candidate(impressions):
    if not isinstance(impressions, str) or not impressions.strip():
        return None

    first = impressions.split()[0]
    return first.rsplit("-", 1)[0]

sample["candidate_news_id"] = (
    sample["impressions"].apply(first_candidate)
)

# Calculate history length
sample["hist_len"] = (
    sample["history"].apply(get_hist_len)
)

# Calculate category matching
sample["cat_match"] = sample.apply(
    lambda row: get_cat_match(
        row["history"],
        row["candidate_news_id"]
    ),
    axis=1
)

# Calculate semantic similarity
sample["user_article_sim"] = [
    get_user_article_sim(history, news_id)
    for history, news_id in zip(
        sample["history"],
        sample["candidate_news_id"]
    )
]

# Display results
display(
    sample[
        [
            "impression_id",
            "candidate_news_id",
            "hist_len",
            "cat_match",
            "user_article_sim"
        ]
    ]
)

# Validate features
assert (sample["hist_len"] >= 0).all()
assert sample["cat_match"].between(0, 1).all()
assert sample["user_article_sim"].between(-1, 1).all()
assert sample["slate_exposure_pct"].between(0, 1).all()
assert sample["user_article_sim"].notna().all()

print("All user-history and similarity checks passed!")

,impression_id,candidate_news_id,hist_len,cat_match,user_article_sim
0,1,N78206,16,0.125000,0.253448
1,2,N47996,24,0.000000,0.064297
2,3,N103852,16,0.187500,0.064860
3,4,N38902,13,0.000000,0.029565
4,6,N32154,15,0.133333,0.239743
5,7,N12681,4,0.250000,0.061404
6,8,N98178,13,0.384615,0.405308
7,9,N49168,11,0.090909,0.116880
8,10,N92691,47,0.510638,0.459138
9,13,N95525,15,0.400000,0.143915


All user-history and similarity checks passed!


In [ ]:
# ==========================================================
# 13. VALIDATE HISTORICAL EXPOSURE FEATURE
# ==========================================================

fit_candidates_file = pq.ParquetFile(
    PROCESSED_DIR / "fit_candidates_sampled.parquet"
)

# Sample across row groups to include different periods
rng = np.random.default_rng(42)

row_groups = rng.choice(
    fit_candidates_file.num_row_groups,
    size=min(8, fit_candidates_file.num_row_groups),
    replace=False
)

samples = []

for rg in row_groups:
    df = fit_candidates_file.read_row_group(
        int(rg),
        columns=[
            "impression_id",
            "news_id",
            "time",
            "prior_exposure_count",
            "cold_article"
        ]
    ).to_pandas()

    samples.append(
        df.sample(
            n=min(100, len(df)),
            random_state=42
        )
    )

exposure_sample = pd.concat(samples, ignore_index=True)

# Convert exposure count into a smaller numerical scale
exposure_sample["log_prior_exposure_count"] = np.log1p(
    exposure_sample["prior_exposure_count"]
)

display(exposure_sample.head(10))

# Check valid values
assert (exposure_sample["prior_exposure_count"] >= 0).all()
assert np.isfinite(
    exposure_sample["log_prior_exposure_count"]
).all()

print("Historical exposure feature validation passed!")
print("Min exposure:", exposure_sample["prior_exposure_count"].min())
print("Max exposure:", exposure_sample["prior_exposure_count"].max())


,impression_id,news_id,time,prior_exposure_count,cold_article,log_prior_exposure_count
0,1261612,N94777,2019-11-13 22:02:05,0,1,0.000000
1,1395136,N57497,2019-11-13 21:40:20,15,0,2.772589
2,274197,N123683,2019-11-13 21:17:24,8,1,2.197225
3,738354,N79233,2019-11-13 21:19:18,0,1,0.000000
4,270303,N2066,2019-11-13 21:45:00,166,0,5.117994
5,822598,N123683,2019-11-13 22:42:12,8,1,2.197225
6,1423733,N53474,2019-11-13 20:36:07,3,1,1.386294
7,518202,N123683,2019-11-13 21:07:52,8,1,2.197225
8,2181967,N42876,2019-11-13 21:29:20,3,1,1.386294
9,99343,N108057,2019-11-13 22:42:01,0,1,0.000000


Historical exposure feature validation passed!
Min exposure: 0
Max exposure: 185815


In [ ]:
# ==========================================================
# 14. FIND EACH ARTICLE'S FIRST OBSERVED IMPRESSION
# ==========================================================

first_seen_path = (
    PROCESSED_DIR / "step3_article_first_seen.parquet"
)

if first_seen_path.exists():
    print("Existing first-seen file found. Loading...")

    article_first_seen = pd.read_parquet(first_seen_path)

else:
    first_seen = {}

    # Process all three chronological data splits
    for split in ["fit", "tune", "eval"]:

        print(f"\nProcessing {split.upper()} impressions...")

        file = pq.ParquetFile(
            PROCESSED_DIR / f"{split}_impressions.parquet"
        )

        for batch_num, batch in enumerate(
            file.iter_batches(
                batch_size=5000,
                columns=["time", "impressions"]
            ),
            start=1
        ):

            df = batch.to_pandas()

            # Convert impression times to datetime
            times = pd.to_datetime(
                df["time"],
                errors="coerce",
                utc=True
            )

            for time, impressions in zip(
                times, df["impressions"]
            ):

                if pd.isna(time):
                    continue

                if not isinstance(impressions, str):
                    continue

                time_ns = time.value

                # Extract every candidate article
                for item in impressions.split():

                    news_id = item.rsplit("-", 1)[0]

                    # Save the earliest observed display time
                    previous = first_seen.get(news_id)

                    if previous is None or time_ns < previous:
                        first_seen[news_id] = time_ns

            if batch_num % 50 == 0:
                print(
                    f"{split.upper()}: "
                    f"{batch_num} batches processed"
                )

    # Convert lookup to DataFrame
    article_first_seen = pd.DataFrame({
        "news_id": list(first_seen.keys()),
        "first_seen_time": pd.to_datetime(
            list(first_seen.values()),
            unit="ns",
            utc=True
        )
    })

    # Save so we do not need to scan everything again
    article_first_seen.to_parquet(
        first_seen_path,
        index=False
    )

    print("\nFirst-seen timestamps saved!")

# Validation
assert article_first_seen["news_id"].is_unique
assert article_first_seen["first_seen_time"].notna().all()

print("\n=== FIRST-SEEN SUMMARY ===")
print("Unique articles:", len(article_first_seen))
print(
    "Earliest appearance:",
    article_first_seen["first_seen_time"].min()
)
print(
    "Latest first appearance:",
    article_first_seen["first_seen_time"].max()
)

display(article_first_seen.head(10))



Processing FIT impressions...
FIT: 50 batches processed
FIT: 100 batches processed
FIT: 150 batches processed
FIT: 200 batches processed
FIT: 250 batches processed
FIT: 300 batches processed
FIT: 350 batches processed

Processing TUNE impressions...
TUNE: 50 batches processed

Processing EVAL impressions...
EVAL: 50 batches processed

First-seen timestamps saved!

=== FIRST-SEEN SUMMARY ===
Unique articles: 29309
Earliest appearance: 2019-11-09 00:00:00+00:00
Latest first appearance: 2019-11-15 23:50:54+00:00


,news_id,first_seen_time
0,N78206,2019-11-09 03:38:53+00:00
1,N26368,2019-11-09 16:02:57+00:00
2,N7578,2019-11-09 16:00:49+00:00
3,N58592,2019-11-09 00:19:27+00:00
4,N19858,2019-11-09 04:21:04+00:00
5,N58258,2019-11-09 16:01:33+00:00
6,N18478,2019-11-09 16:25:35+00:00
7,N2591,2019-11-09 16:54:47+00:00
8,N97778,2019-11-09 19:26:08+00:00
9,N32954,2019-11-09 17:52:50+00:00


In [ ]:

# ==========================================================
# 15. CALCULATE AND VALIDATE ARTICLE AGE
# ==========================================================

# Use the candidate sample from our earlier exposure test
age_sample = exposure_sample.copy()

# Attach the first-seen timestamp to each article
age_sample = age_sample.merge(
    article_first_seen,
    on="news_id",
    how="left",
    validate="many_to_one"
)

# Convert impression timestamps to datetime
age_sample["time"] = pd.to_datetime(
    age_sample["time"],
    errors="coerce",
    utc=True
)

# Check that every candidate has valid timestamps
assert age_sample["time"].notna().all()
assert age_sample["first_seen_time"].notna().all(), (
    "Some candidate articles have no first-seen timestamp."
)

# Calculate hours since first observed display
age_sample["article_age_hours"] = (
    age_sample["time"] - age_sample["first_seen_time"]
).dt.total_seconds() / 3600

# Confirm that there are no negative ages
assert (age_sample["article_age_hours"] >= 0).all()

# Apply log transformation for later MLP training
age_sample["log_article_age_hours"] = np.log1p(
    age_sample["article_age_hours"]
)

# Validate the transformed feature
assert np.isfinite(
    age_sample["log_article_age_hours"]
).all()

display(
    age_sample[
        [
            "news_id",
            "prior_exposure_count",
            "first_seen_time",
            "time",
            "article_age_hours",
            "log_article_age_hours"
        ]
    ].head(10)
)

print("Article age validation passed!")
print("Min article age:", age_sample["article_age_hours"].min())
print("Max article age:", age_sample["article_age_hours"].max())
print("Mean article age:", age_sample["article_age_hours"].mean())


,news_id,prior_exposure_count,first_seen_time,time,article_age_hours,log_article_age_hours
0,N94777,0,2019-11-13 16:02:47+00:00,2019-11-13 22:02:05+00:00,5.988333,1.944242
1,N57497,15,2019-11-12 16:13:32+00:00,2019-11-13 21:40:20+00:00,29.446667,3.415977
2,N123683,8,2019-11-12 16:26:12+00:00,2019-11-13 21:17:24+00:00,28.853333,3.396297
3,N79233,0,2019-11-13 10:27:06+00:00,2019-11-13 21:19:18+00:00,10.870000,2.474014
4,N2066,166,2019-11-12 07:53:02+00:00,2019-11-13 21:45:00+00:00,37.866111,3.660123
5,N123683,8,2019-11-12 16:26:12+00:00,2019-11-13 22:42:12+00:00,30.266667,3.442553
6,N53474,3,2019-11-12 16:26:24+00:00,2019-11-13 20:36:07+00:00,28.161944,3.372865
7,N123683,8,2019-11-12 16:26:12+00:00,2019-11-13 21:07:52+00:00,28.694444,3.390960
8,N42876,3,2019-11-12 16:54:55+00:00,2019-11-13 21:29:20+00:00,28.573611,3.386882
9,N108057,0,2019-11-13 16:02:45+00:00,2019-11-13 22:42:01+00:00,6.654444,2.035286


Article age validation passed!
Min article age: 0.4513888888888889
Max article age: 117.21583333333334
Mean article age: 25.822285416666663


In [ ]:
# ==========================================================
# 16. PREPARE CANDIDATE-LEVEL FEATURES
# ==========================================================

# Article first-appearance lookup
first_seen_lookup = (
    article_first_seen
    .set_index("news_id")["first_seen_time"]
)

def prepare_candidate_features(df):

    df = df.copy()

    # 1. Convert category and subcategory to integer indices
    df["category_idx"] = (
        df["category"]
        .map(category_to_idx)
        .fillna(0)
        .astype("int32")
    )

    df["subcategory_idx"] = (
        df["subcategory"]
        .map(subcategory_to_idx)
        .fillna(0)
        .astype("int32")
    )

    # 2. Log-transform previous exposure counts
    assert df["prior_exposure_count"].notna().all()
    assert (df["prior_exposure_count"] >= 0).all()

    df["log_prior_exposure_count"] = np.log1p(
        df["prior_exposure_count"]
    )

    # 3. Calculate article age
    df["time"] = pd.to_datetime(
        df["time"],
        utc=True,
        errors="coerce"
    )

    first_seen = df["news_id"].map(first_seen_lookup)

    assert df["time"].notna().all()
    assert first_seen.notna().all(), (
        "Some candidate articles have no first-seen time."
    )

    df["article_age_hours"] = (
        df["time"] - first_seen
    ).dt.total_seconds() / 3600

    assert (df["article_age_hours"] >= 0).all()

    # 4. Log-transform article age
    df["log_article_age_hours"] = np.log1p(
        df["article_age_hours"]
    )

    return df


print("Candidate feature preparation function ready!")


Candidate feature preparation function ready!


In [ ]:
# ==========================================================
# 17. VALIDATE COMBINED CANDIDATE FEATURES
# ==========================================================

# Read 1,000 candidate observations from FIT
test_batch = next(
    pq.ParquetFile(
        PROCESSED_DIR / "fit_candidates_sampled.parquet"
    ).iter_batches(batch_size=1000)
).to_pandas()

# Apply our reusable function
test_features = prepare_candidate_features(test_batch)

display(
    test_features[
        [
            "impression_id",
            "news_id",
            "category_idx",
            "subcategory_idx",
            "log_prior_exposure_count",
            "article_age_hours",
            "log_article_age_hours"
        ]
    ].head(10)
)

# Validate
numeric_cols = [
    "log_prior_exposure_count",
    "article_age_hours",
    "log_article_age_hours"
]

assert np.isfinite(
    test_features[numeric_cols].to_numpy()
).all()

assert (test_features["category_idx"] >= 0).all()
assert (test_features["subcategory_idx"] >= 0).all()

print("Combined candidate features validation passed!")


,impression_id,news_id,category_idx,subcategory_idx,log_prior_exposure_count,article_age_hours,log_article_age_hours
0,1000141,N93672,12,31,0.0,1.406944,0.878358
1,1000141,N31854,13,193,0.0,1.382778,0.868267
2,1000141,N2198,1,7,0.0,1.426111,0.886290
3,1000141,N104430,7,95,0.0,1.421389,0.884341
4,1000141,N103397,10,149,0.0,1.425556,0.886061
5,100018,N103397,10,149,0.0,1.659444,0.978117
6,100018,N59303,3,58,0.0,1.659444,0.978117
7,100018,N118589,9,137,0.0,1.658056,0.977595
8,100018,N2198,1,7,0.0,1.660000,0.978326
9,100018,N32209,10,155,0.0,1.655278,0.976549


Combined candidate features validation passed!


In [ ]:
# ==========================================================
# 18. MATCH CANDIDATES WITH USER READING HISTORIES
# ==========================================================

# Continue using the 1,000-row candidate test batch
test_features = prepare_candidate_features(test_batch)

# Find the impression IDs we need
remaining_ids = set(
    test_features["impression_id"].astype(str)
)

history_lookup = {}

impressions_file = pq.ParquetFile(
    PROCESSED_DIR / "fit_impressions.parquet"
)

# Search for matching impressions
for batch in impressions_file.iter_batches(
    batch_size=50000,
    columns=["impression_id", "history"]
):

    df = batch.to_pandas()

    df["_id"] = df["impression_id"].astype(str)

    matched = df[df["_id"].isin(remaining_ids)]

    history_lookup.update(
        zip(matched["_id"], matched["history"])
    )

    remaining_ids -= set(matched["_id"])

    if not remaining_ids:
        break

# Check that all impressions were found
assert not remaining_ids, (
    f"Missing {len(remaining_ids)} impression IDs"
)

# Attach reading histories to candidate observations
test_features["history"] = (
    test_features["impression_id"]
    .astype(str)
    .map(history_lookup)
    .fillna("")
)

print("Candidate rows:", len(test_features))
print("Unique impressions:", len(history_lookup))
print("All candidate histories matched!")


Candidate rows: 1000
Unique impressions: 139
All candidate histories matched!


In [ ]:

# ==========================================================
# 19. COMBINE PERSONALISED AND ARTICLE FEATURES
# ==========================================================

# Compute history profiles once per impression
user_profiles = {
    impression_id: get_user_profile(history)
    for impression_id, history in history_lookup.items()
}

# History length
history_lengths = {
    impression_id: get_hist_len(history)
    for impression_id, history in history_lookup.items()
}

test_features["hist_len"] = (
    test_features["impression_id"]
    .astype(str)
    .map(history_lengths)
    .astype("int32")
)

# Category match
test_features["cat_match"] = [
    get_cat_match(history, news_id)
    for history, news_id in zip(
        test_features["history"],
        test_features["news_id"]
    )
]

# User-article similarity
def similarity_from_profile(impression_id, news_id):
    profile = user_profiles.get(str(impression_id))
    article_idx = news_to_embedding_idx.get(str(news_id))

    if profile is None or article_idx is None:
        return 0.0

    return float(np.clip(
        np.dot(profile, news_embeddings[article_idx]),
        -1.0,
        1.0
    ))

test_features["user_article_sim"] = [
    similarity_from_profile(imp_id, news_id)
    for imp_id, news_id in zip(
        test_features["impression_id"],
        test_features["news_id"]
    )
]

# Display the seven final MLP features
feature_columns = [
    "category_idx",
    "subcategory_idx",
    "hist_len",
    "cat_match",
    "user_article_sim",
    "log_prior_exposure_count",
    "log_article_age_hours"
]

display(
    test_features[
        ["impression_id", "news_id", "label"]
        + feature_columns
    ].head(10)
)

# Validate all numerical features
assert np.isfinite(
    test_features[feature_columns].to_numpy()
).all()

assert test_features["cat_match"].between(0, 1).all()
assert test_features["user_article_sim"].between(-1, 1).all()

print("All seven MLP features combined successfully!")


,impression_id,news_id,label,category_idx,subcategory_idx,hist_len,cat_match,user_article_sim,log_prior_exposure_count,log_article_age_hours
0,1000141,N93672,1,12,31,8,0.250,0.221685,0.0,0.878358
1,1000141,N31854,0,13,193,8,0.125,0.123348,0.0,0.868267
2,1000141,N2198,0,1,7,8,0.000,0.053291,0.0,0.886290
3,1000141,N104430,0,7,95,8,0.250,0.109273,0.0,0.884341
4,1000141,N103397,0,10,149,8,0.125,0.187800,0.0,0.886061
5,100018,N103397,1,10,149,5,0.800,0.264914,0.0,0.978117
6,100018,N59303,0,3,58,5,0.000,0.022502,0.0,0.978117
7,100018,N118589,0,9,137,5,0.000,0.068552,0.0,0.977595
8,100018,N2198,0,1,7,5,0.000,-0.052812,0.0,0.978326
9,100018,N32209,0,10,155,5,0.800,0.091976,0.0,0.976549


All seven MLP features combined successfully!


In [ ]:
# ==========================================================
# 20. OPTIMISED PERSONALISED FEATURE CALCULATIONS
# ==========================================================

from collections import Counter

def add_personalised_features(df):

    df = df.copy().reset_index(drop=True)

    n = len(df)

    hist_lengths = np.zeros(n, dtype=np.int32)
    cat_matches = np.zeros(n, dtype=np.float32)
    similarities = np.zeros(n, dtype=np.float32)

    # Process each impression once
    for impression_id, group in df.groupby(
        "impression_id", sort=False
    ):

        positions = group.index.to_numpy()
        history = group["history"].iloc[0]

        history_ids = parse_history(history)
        hist_len = len(history_ids)

        hist_lengths[positions] = hist_len

        # Count categories in the user's reading history
        category_counts = Counter(
            news_to_category.get(nid)
            for nid in history_ids
            if pd.notna(news_to_category.get(nid))
        )

        # Category matching for each candidate
        if hist_len > 0:
            cat_matches[positions] = [
                (
                    category_counts.get(
                        news_to_category.get(news_id), 0
                    ) / hist_len
                    if pd.notna(news_to_category.get(news_id))
                    else 0.0
                )
                for news_id in group["news_id"]
            ]

        # Generate user profile only once per impression
        user_profile = get_user_profile(history)

        if user_profile is not None:

            candidate_indices = [
                news_to_embedding_idx.get(str(news_id))
                for news_id in group["news_id"]
            ]

            valid = np.array([
                idx is not None
                for idx in candidate_indices
            ])

            if valid.any():

                indices = [
                    idx for idx in candidate_indices
                    if idx is not None
                ]

                candidate_vectors = np.asarray(
                    news_embeddings[indices],
                    dtype=np.float32
                )

                similarities[positions[valid]] = np.clip(
                    candidate_vectors @ user_profile,
                    -1.0, 1.0
                )

    df["hist_len"] = hist_lengths
    df["cat_match"] = cat_matches
    df["user_article_sim"] = similarities

    return df


print("Optimised personalised feature function ready!")


Optimised personalised feature function ready!


In [ ]:
# ==========================================================
# 21. VALIDATE OPTIMISED FEATURES
# ==========================================================

optimised_test = add_personalised_features(test_features)

check_columns = [
    "hist_len",
    "cat_match",
    "user_article_sim"
]

for col in check_columns:

    matching = np.allclose(
        test_features[col].to_numpy(),
        optimised_test[col].to_numpy(),
        rtol=1e-5,
        atol=1e-5
    )

    print(f"{col}: {'MATCH' if matching else 'MISMATCH'}")

    assert matching, f"{col} does not match!"

print("\nOptimised calculations match original results!")


hist_len: MATCH
cat_match: MATCH
user_article_sim: MATCH

Optimised calculations match original results!


In [ ]:
#%pip install duckdb

   ---------------------------------------- 0.0/15.4 MB ? eta -:--:--
   -------------- ------------------------- 5.5/15.4 MB 30.5 MB/s eta 0:00:01
   ------------------------------------ --- 14.2/15.4 MB 35.5 MB/s eta 0:00:01
   ---------------------------------------  15.2/15.4 MB 35.4 MB/s eta 0:00:01
   ---------------------------------------- 15.4/15.4 MB 21.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# ==========================================================
# 22. TEST FULL-DATA JOIN PIPELINE
# ==========================================================

import duckdb

# Connect to DuckDB
con = duckdb.connect()

candidate_path = str(
    PROCESSED_DIR / "fit_candidates_sampled.parquet"
)

impression_path = str(
    PROCESSED_DIR / "fit_impressions.parquet"
)

# Join candidate observations with their reading histories
# Limit candidates to 1,000 for this test
query = """
WITH sample_candidates AS (
    SELECT *
    FROM read_parquet(?)
    LIMIT 1000
)

SELECT
    c.*,
    i.history,
    i.impression_id AS matched_impression_id

FROM sample_candidates c

LEFT JOIN read_parquet(?) i
    ON c.impression_id = i.impression_id
"""

joined_sample = con.execute(
    query,
    [candidate_path, impression_path]
).df()

# Ensure every candidate matched exactly one impression
assert len(joined_sample) == 1000
assert joined_sample["matched_impression_id"].notna().all()

# Handle users with no recorded click history
joined_sample["history"] = (
    joined_sample["history"].fillna("")
)

# Remove temporary validation column
joined_sample = joined_sample.drop(
    columns=["matched_impression_id"]
)

# Calculate article-level features
joined_sample = prepare_candidate_features(
    joined_sample
)

# Calculate personalised features
joined_sample = add_personalised_features(
    joined_sample
)

# Final feature columns
feature_columns = [
    "category_idx",
    "subcategory_idx",
    "hist_len",
    "cat_match",
    "user_article_sim",
    "log_prior_exposure_count",
    "log_article_age_hours"
]

# Validate
assert np.isfinite(
    joined_sample[feature_columns].to_numpy()
).all()

assert joined_sample["label"].isin([0, 1]).all()

print("Total candidate rows:", len(joined_sample))
print("Unique impressions:", joined_sample["impression_id"].nunique())
print("Number of MLP features:", len(feature_columns))

display(
    joined_sample[
        ["impression_id", "news_id", "label"]
        + feature_columns
    ].head(10)
)

print("\nFull-data pipeline test passed!")


Total candidate rows: 1000
Unique impressions: 139
Number of MLP features: 7


,impression_id,news_id,label,category_idx,subcategory_idx,hist_len,cat_match,user_article_sim,log_prior_exposure_count,log_article_age_hours
0,100018,N32209,0,10,155,5,0.800000,0.091976,0.0,0.976549
1,100060,N110664,0,12,29,4,0.250000,0.130543,0.0,0.737563
2,100094,N76042,0,10,144,208,0.341346,0.200774,0.0,0.342604
3,100104,N65783,0,14,198,9,0.111111,0.174427,0.0,1.539015
4,100196,N32209,0,10,155,7,0.285714,0.108312,0.0,0.752726
5,100204,N6865,0,8,128,12,0.000000,0.076437,0.0,1.173170
6,100209,N37460,0,13,193,33,0.030303,0.237267,0.0,1.014828
7,100927,N10470,0,13,194,3,0.000000,0.080353,0.0,1.014425
8,100937,N118589,0,9,137,5,0.000000,0.121271,0.0,1.200718
9,100999,N101119,0,1,7,1,0.000000,-0.082116,0.0,1.422104



Full-data pipeline test passed!


In [2]:

# ==========================================================
# EDITED FEATURE: WITHIN-SLATE EXPOSURE PERCENTILE
# ==========================================================

import numpy as np

def calculate_slate_exposure_pct(news_ids, prior_counts):
    """
    Rank articles by their previous-day exposure counts
    within the ORIGINAL full impression slate.

    Returns exposure percentiles between 0 and 1.
    """

    n = len(news_ids)

    if n == 0:
        return {}

    if n == 1:
        return {str(news_ids[0]): 0.5}

    exposures = np.array([
        prior_counts.get(str(nid), 0)
        for nid in news_ids
    ])

    # Equal exposure counts receive equal average ranks
    _, inverse, counts = np.unique(
        exposures,
        return_inverse=True,
        return_counts=True
    )

    start_ranks = np.cumsum(counts) - counts
    average_ranks = start_ranks + (counts - 1) / 2

    percentiles = average_ranks[inverse] / (n - 1)

    return dict(zip(
        map(str, news_ids),
        percentiles.astype(np.float32)
    ))


# Test with five articles
test_news = ["A", "B", "C", "D", "E"]

test_counts = {
    "A": 10,
    "B": 50,
    "C": 100,
    "D": 500,
    "E": 1000
}

result = calculate_slate_exposure_pct(
    test_news, test_counts
)

print("Exposure percentiles:", result)

assert np.allclose(
    list(result.values()),
    [0, 0.25, 0.5, 0.75, 1]
)

# Test ties: all zero exposures should return 0.5
zero_result = calculate_slate_exposure_pct(
    ["A", "B", "C"], {}
)
assert all(np.isclose(v, 0.5) for v in zero_result.values())

print("Exposure percentile function PASSED!")


Exposure percentiles: {'A': np.float32(0.0), 'B': np.float32(0.25), 'C': np.float32(0.5), 'D': np.float32(0.75), 'E': np.float32(1.0)}
Exposure percentile function PASSED!


In [3]:

# ==========================================================
# CALCULATE PREVIOUS-DAY EXPOSURES FROM FULL SLATES
# ==========================================================

from collections import Counter, defaultdict
import pandas as pd
import pyarrow.parquet as pq

# Count displays for each article on each day
daily_exposures = defaultdict(Counter)

for split in ["fit", "tune", "eval"]:

    print(f"\nReading {split.upper()} impressions...", flush=True)

    pf = pq.ParquetFile(
        PROCESSED_DIR / f"{split}_impressions.parquet"
    )

    for batch_num, batch in enumerate(
        pf.iter_batches(
            batch_size=5000,
            columns=["time", "impressions"]
        ),
        start=1
    ):

        df = batch.to_pandas()

        days = pd.to_datetime(
            df["time"], utc=True
        ).dt.date

        for day, slate in zip(days, df["impressions"]):

            if not isinstance(slate, str):
                continue

            news_ids = [
                item.rsplit("-", 1)[0]
                for item in slate.split()
            ]

            daily_exposures[day].update(news_ids)

        if batch_num % 100 == 0:
            print(
                f"{split.upper()}: "
                f"{batch_num * 5000:,} impressions processed",
                flush=True
            )

    print(f"Finished {split.upper()}!", flush=True)


# Calculate exposure counts from EARLIER days only
prior_by_day = {}
running_counts = Counter()

for day in sorted(daily_exposures):

    # Save counts BEFORE adding the current day's displays
    prior_by_day[day] = running_counts.copy()

    # Current-day displays only count from tomorrow
    running_counts.update(daily_exposures[day])


# Validation
first_day = min(prior_by_day)

assert sum(prior_by_day[first_day].values()) == 0

print("\n=== PRIOR EXPOSURE SUMMARY ===")

for day in sorted(prior_by_day):
    print(
        f"{day}: "
        f"{sum(prior_by_day[day].values()):,} "
        "previous displays"
    )

print("\nPrevious-day exposure calculation PASSED!")



Reading FIT impressions...
FIT: 500,000 impressions processed
FIT: 1,000,000 impressions processed
FIT: 1,500,000 impressions processed
Finished FIT!

Reading TUNE impressions...
Finished TUNE!

Reading EVAL impressions...
Finished EVAL!

=== PRIOR EXPOSURE SUMMARY ===
2019-11-09: 0 previous displays
2019-11-10: 7,544,547 previous displays
2019-11-11: 15,040,760 previous displays
2019-11-12: 31,623,976 previous displays
2019-11-13: 48,368,810 previous displays
2019-11-14: 66,107,268 previous displays
2019-11-15: 83,507,374 previous displays

Previous-day exposure calculation PASSED!


In [4]:

# ==========================================================
# APPLY WITHIN-SLATE EXPOSURE PERCENTILE
# ==========================================================

def add_slate_exposure_percentile(df):
    """
    Add each candidate article's exposure percentile
    within its ORIGINAL full impression slate.
    """

    df = df.copy().reset_index(drop=True)

    results = np.zeros(len(df), dtype=np.float32)

    for impression_id, group in df.groupby(
        "impression_id", sort=False
    ):
        # Original full slate (before FIT negative sampling)
        full_slate = group["full_slate"].iloc[0]

        if not isinstance(full_slate, str):
            raise ValueError(
                f"Missing full slate for impression {impression_id}"
            )

        full_news_ids = [
            item.rsplit("-", 1)[0]
            for item in full_slate.split()
        ]

        # Only use exposure counts from previous days
        day = pd.to_datetime(
            group["time"].iloc[0], utc=True
        ).date()

        exposure_lookup = calculate_slate_exposure_pct(
            full_news_ids,
            prior_by_day[day]
        )

        # Match percentiles to the candidate rows
        candidate_ids = group["news_id"].astype(str).tolist()

        missing = set(candidate_ids) - set(exposure_lookup)
        if missing:
            raise ValueError(
                f"Candidates missing from original slate: {missing}"
            )

        results[group.index.to_numpy()] = [
            exposure_lookup[nid]
            for nid in candidate_ids
        ]

    df["slate_exposure_pct"] = results

    assert df["slate_exposure_pct"].between(0, 1).all()

    return df


# ==========================================================
# TEST USING A REAL TUNE IMPRESSION
# ==========================================================

pf = pq.ParquetFile(
    PROCESSED_DIR / "tune_impressions.parquet"
)

sample = next(
    pf.iter_batches(
        batch_size=1,
        columns=["impression_id", "time", "impressions"]
    )
).to_pandas().iloc[0]

full_news_ids = [
    item.rsplit("-", 1)[0]
    for item in sample["impressions"].split()
]

# Use just five candidate rows for testing,
# while calculating ranks against the FULL slate.
test_ids = full_news_ids[:5]

test_df = pd.DataFrame({
    "impression_id": [sample["impression_id"]] * len(test_ids),
    "news_id": test_ids,
    "time": [sample["time"]] * len(test_ids),
    "full_slate": [sample["impressions"]] * len(test_ids)
})

test_result = add_slate_exposure_percentile(test_df)

display(test_result[["news_id", "slate_exposure_pct"]])

print("Full slate size:", len(full_news_ids))
print("Candidate rows tested:", len(test_result))
print("Exposure percentile calculation PASSED!")


,news_id,slate_exposure_pct
0,N76209,0.437500
1,N48841,0.105469
2,N67937,0.773438
3,N62235,0.625000
4,N6307,0.890625


Full slate size: 129
Candidate rows tested: 5
Exposure percentile calculation PASSED!


In [5]:
# ==========================================================
# 23. GENERATE COMPLETE MLP FEATURE DATASETS
# ==========================================================

import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np

feature_columns = [
    "category_idx",
    "subcategory_idx",
    "hist_len",
    "cat_match",
    "user_article_sim",
    "slate_exposure_pct",
    "log_article_age_hours"
]

def generate_feature_dataset(split, batch_size=20000):

    candidate_filename = {
        "fit": "fit_candidates_sampled.parquet",
        "tune": "tune_candidates.parquet",
        "eval": "eval_candidates.parquet"
    }[split]

    candidate_path = PROCESSED_DIR / candidate_filename
    impression_path = (
        PROCESSED_DIR / f"{split}_impressions.parquet"
    )

    output_path = (
        PROCESSED_DIR / f"features_{split}.parquet"
    )

    # Save to a temporary file until everything succeeds
    temp_path = (
        PROCESSED_DIR / f"features_{split}.partial.parquet"
    )

    if output_path.exists():
        print(f"Updating existing {output_path.name}...")
        print("Old file will remain until the new file passes validation.")

    expected_rows = pq.ParquetFile(
        candidate_path
    ).metadata.num_rows

    print(f"Generating {split.upper()} features...")
    print(f"Expected rows: {expected_rows:,}")

    # Join P1 candidate data with recorded user histories
    query = """
        SELECT
            c.*,
            i.history,
            i.impressions AS full_slate,
            i.impression_id AS matched_impression_id
        FROM read_parquet(?) AS c
        LEFT JOIN read_parquet(?) AS i
            ON c.impression_id = i.impression_id
    """

    con = duckdb.connect()

    reader = con.sql(
        query,
        params=[str(candidate_path), str(impression_path)]
    ).to_arrow_reader(batch_size=batch_size)

    keep_columns = [
        "impression_id",
        "user_id",
        "news_id",
        "time",
        "label",
        "cold_article",
        "category_idx",
        "subcategory_idx",
        "hist_len",
        "cat_match",
        "user_article_sim",
        "slate_exposure_pct",
        "log_article_age_hours"
    ]

    writer = None
    total_rows = 0

    try:
        for batch_number, batch in enumerate(reader, start=1):

            df = batch.to_pandas()

            # Check that candidates matched an impression
            assert df["matched_impression_id"].notna().all(), (
                "Some candidate impressions were not matched."
            )

            df["history"] = df["history"].fillna("")

            # Calculate article features
            df = prepare_candidate_features(df)

            # Calculate personalised features
            df = add_slate_exposure_percentile(df)

            # Calculate exposure percentile using original full slate
            df = add_slate_exposure_percentile(df)

            # Keep only fields needed by the model/evaluation
            output = df[keep_columns].copy()

            # Validate numerical features
            assert np.isfinite(
                output[feature_columns].to_numpy()
            ).all()

            assert output["label"].isin([0, 1]).all()

            # Convert to Arrow for Parquet writing
            table = pa.Table.from_pandas(
                output,
                preserve_index=False
            )

            if writer is None:
                writer = pq.ParquetWriter(
                    temp_path,
                    table.schema,
                    compression="snappy"
                )

            writer.write_table(table)

            total_rows += len(output)

            if batch_number % 25 == 0:
                print(
                    f"{split.upper()}: "
                    f"{total_rows:,} rows processed"
                )

    finally:
        if writer is not None:
            writer.close()
        con.close()

    # Check that we haven't lost or duplicated candidates
    assert total_rows == expected_rows, (
        f"Expected {expected_rows:,} rows, "
        f"but generated {total_rows:,}."
    )

    # Validate saved file
    saved_rows = pq.ParquetFile(
        temp_path
    ).metadata.num_rows

    assert saved_rows == expected_rows

    # Only make the final file available after validation
    temp_path.replace(output_path)

    print(f"\n{split.upper()} generation completed!")
    print(f"Saved rows: {saved_rows:,}")
    print(f"Saved to: {output_path}")


In [ ]:
#FIT
generate_feature_dataset("fit")

Generating FIT features...
Expected rows: 12,894,210
FIT: 500,000 rows processed
FIT: 1,000,000 rows processed
FIT: 1,500,000 rows processed
FIT: 2,000,000 rows processed
FIT: 2,500,000 rows processed
FIT: 3,000,000 rows processed
FIT: 3,500,000 rows processed
FIT: 4,000,000 rows processed
FIT: 4,500,000 rows processed
FIT: 5,000,000 rows processed
FIT: 5,500,000 rows processed
FIT: 6,000,000 rows processed
FIT: 6,500,000 rows processed
FIT: 7,000,000 rows processed
FIT: 7,500,000 rows processed
FIT: 8,000,000 rows processed
FIT: 8,500,000 rows processed
FIT: 9,000,000 rows processed
FIT: 9,500,000 rows processed
FIT: 10,000,000 rows processed
FIT: 10,500,000 rows processed
FIT: 11,000,000 rows processed
FIT: 11,500,000 rows processed
FIT: 12,000,000 rows processed
FIT: 12,433,123 rows processed
FIT: 12,876,937 rows processed

FIT generation completed!
Saved rows: 12,894,210
Saved to: G:\.shortcut-targets-by-id\1D4Db1s7DJ5PVK9CFiV-pklpCW5dr1uOM\BT4222_Project\data\processed\features_fi

In [ ]:
# ==========================================================
# 25. VALIDATE SAVED FIT FEATURE DATASET
# ==========================================================

fit_path = PROCESSED_DIR / "features_fit.parquet"

fit_file = pq.ParquetFile(fit_path)

required_columns = [
    "impression_id",
    "news_id",
    "label",
    "cold_article",
    "category_idx",
    "subcategory_idx",
    "hist_len",
    "cat_match",
    "user_article_sim",
    "slate_exposure_pct",
    "log_article_age_hours"
]

# 1. Check total rows and columns
expected_rows = pq.ParquetFile(
    PROCESSED_DIR / "fit_candidates_sampled.parquet"
).metadata.num_rows

assert fit_file.metadata.num_rows == expected_rows

assert set(required_columns).issubset(
    fit_file.schema_arrow.names
)

# 2. Sample from the beginning, middle and end
group_indices = sorted(set([
    0,
    fit_file.num_row_groups // 2,
    fit_file.num_row_groups - 1
]))

samples = []

for group_idx in group_indices:
    df = fit_file.read_row_group(group_idx).to_pandas()

    samples.append(
        df.sample(
            n=min(1000, len(df)),
            random_state=42
        )
    )

validation_sample = pd.concat(
    samples,
    ignore_index=True
)

# 3. Check feature values
assert validation_sample["label"].isin([0, 1]).all()

assert validation_sample["cat_match"].between(0, 1).all()

assert validation_sample["user_article_sim"].between(-1, 1).all()

assert (validation_sample["hist_len"] >= 0).all()

assert validation_sample["slate_exposure_pct"].between(0, 1).all()

assert np.isfinite(
    validation_sample[feature_columns].to_numpy()
).all()

# 4. Summary
print("=== FIT DATASET VALIDATION ===")
print("Total rows:", f"{fit_file.metadata.num_rows:,}")
print("Number of MLP features:", len(feature_columns))
print("Sample rows checked:", len(validation_sample))

display(
    validation_sample[
        ["impression_id", "news_id", "label"]
        + feature_columns
    ].head(10)
)

print("\nFIT sample validation passed!")


=== FIT DATASET VALIDATION ===
Total rows: 12,894,210
Number of MLP features: 7
Sample rows checked: 3000


,impression_id,news_id,label,category_idx,subcategory_idx,hist_len,cat_match,user_article_sim,log_prior_exposure_count,log_article_age_hours
0,1228129,N19134,0,10,153,67,0.850746,0.366859,0.0,1.536568
1,1044968,N24017,1,1,12,45,0.066667,0.243192,0.0,1.133908
2,1181795,N2198,0,1,7,8,0.375000,0.323401,0.0,1.442662
3,1023581,N121426,0,12,29,6,0.333333,0.152849,0.0,1.460938
4,1290054,N104437,1,10,144,7,0.571429,0.195216,0.0,0.513323
5,1252143,N115958,0,7,210,15,0.133333,0.114082,0.0,1.367083
6,1236636,N67004,0,10,153,70,0.428571,0.291496,0.0,1.027136
7,1062986,N103397,1,10,149,3,1.000000,0.082612,0.0,0.519124
8,1415983,N118589,0,9,137,12,0.000000,0.205059,0.0,0.521930
9,1172658,N121042,0,12,79,123,0.065041,0.239914,0.0,1.480215



FIT sample validation passed!


In [ ]:
#TUNE
generate_feature_dataset("tune", batch_size=30000)

Generating TUNE features...
Expected rows: 17,400,106
TUNE: 750,000 rows processed
TUNE: 1,500,000 rows processed
TUNE: 2,250,000 rows processed
TUNE: 3,000,000 rows processed
TUNE: 3,750,000 rows processed
TUNE: 4,500,000 rows processed
TUNE: 5,250,000 rows processed
TUNE: 6,000,000 rows processed
TUNE: 6,750,000 rows processed
TUNE: 7,500,000 rows processed
TUNE: 8,250,000 rows processed
TUNE: 9,000,000 rows processed
TUNE: 9,750,000 rows processed
TUNE: 10,500,000 rows processed
TUNE: 11,250,000 rows processed
TUNE: 12,000,000 rows processed
TUNE: 12,750,000 rows processed
TUNE: 13,500,000 rows processed
TUNE: 14,250,000 rows processed
TUNE: 15,000,000 rows processed
TUNE: 15,531,080 rows processed
TUNE: 16,281,080 rows processed
TUNE: 16,930,105 rows processed

TUNE generation completed!
Saved rows: 17,400,106
Saved to: G:\.shortcut-targets-by-id\1D4Db1s7DJ5PVK9CFiV-pklpCW5dr1uOM\BT4222_Project\data\processed\features_tune.parquet


In [ ]:

# ==========================================================
# 26. VALIDATE SAVED TUNE FEATURE DATASET
# ==========================================================

tune_path = PROCESSED_DIR / "features_tune.parquet"
tune_file = pq.ParquetFile(tune_path)

expected_rows = pq.ParquetFile(
    PROCESSED_DIR / "tune_candidates.parquet"
).metadata.num_rows

# Check row count and feature columns
assert tune_file.metadata.num_rows == expected_rows
assert set(feature_columns).issubset(
    tune_file.schema_arrow.names
)

# Sample rows from different parts of the saved file
group_indices = sorted(set([
    0,
    tune_file.num_row_groups // 2,
    tune_file.num_row_groups - 1
]))

samples = []

for group_idx in group_indices:
    df = tune_file.read_row_group(group_idx).to_pandas()
    samples.append(
        df.sample(n=min(1000, len(df)), random_state=42)
    )

tune_sample = pd.concat(samples, ignore_index=True)

# Validate feature values
assert tune_sample["label"].isin([0, 1]).all()
assert tune_sample["cat_match"].between(0, 1).all()
assert tune_sample["user_article_sim"].between(-1, 1).all()
assert (tune_sample["hist_len"] >= 0).all()
assert tune_sample["slate_exposure_pct"].between(0, 1).all()
assert (tune_sample["log_article_age_hours"] >= 0).all()

assert np.isfinite(
    tune_sample[feature_columns].to_numpy()
).all()

print("=== TUNE DATASET VALIDATION ===")
print("Total rows:", f"{tune_file.metadata.num_rows:,}")
print("Number of MLP features:", len(feature_columns))
print("Sample rows checked:", len(tune_sample))

display(
    tune_sample[
        ["impression_id", "news_id", "label"]
        + feature_columns
    ].head(10)
)

print("\nTUNE sample validation passed!")


=== TUNE DATASET VALIDATION ===
Total rows: 17,400,106
Number of MLP features: 7
Sample rows checked: 3000


,impression_id,news_id,label,category_idx,subcategory_idx,hist_len,cat_match,user_article_sim,log_prior_exposure_count,log_article_age_hours
0,1489741,N8734,0,5,122,75,0.106667,0.261092,7.604396,2.997287
1,1243188,N41388,0,10,151,13,0.230769,0.167048,8.795734,3.469669
2,2066996,N81914,0,10,149,17,0.000000,0.122490,8.655563,2.218811
3,1496429,N42039,0,7,210,46,0.065217,0.215312,8.899321,2.223422
4,731235,N82315,0,4,76,29,0.103448,0.302956,11.375524,4.765405
5,1110921,N72229,0,4,187,64,0.062500,0.117208,9.600421,3.479632
6,661811,N37479,0,5,1,126,0.023810,0.264394,10.534600,3.492941
7,138947,N31095,0,10,155,5,0.200000,0.152341,9.457669,2.206197
8,1766198,N42876,0,2,51,64,0.000000,0.197314,10.478245,3.472079
9,1189486,N123968,0,1,12,14,0.000000,0.260145,11.342647,4.449017



TUNE sample validation passed!


In [ ]:
#EVAL
generate_feature_dataset("eval", batch_size=30000)

Generating EVAL features...
Expected rows: 14,085,557
EVAL: 750,000 rows processed
EVAL: 1,500,000 rows processed
EVAL: 2,250,000 rows processed
EVAL: 3,000,000 rows processed
EVAL: 3,750,000 rows processed
EVAL: 4,500,000 rows processed
EVAL: 5,250,000 rows processed
EVAL: 6,000,000 rows processed
EVAL: 6,750,000 rows processed
EVAL: 7,500,000 rows processed
EVAL: 8,230,032 rows processed
EVAL: 8,964,811 rows processed
EVAL: 9,714,811 rows processed
EVAL: 10,464,811 rows processed
EVAL: 11,214,811 rows processed
EVAL: 11,964,811 rows processed
EVAL: 12,714,811 rows processed
EVAL: 13,444,914 rows processed
EVAL: 13,975,544 rows processed

EVAL generation completed!
Saved rows: 14,085,557
Saved to: G:\.shortcut-targets-by-id\1D4Db1s7DJ5PVK9CFiV-pklpCW5dr1uOM\BT4222_Project\data\processed\features_eval.parquet


In [ ]:

# ==========================================================
# 27. FINAL VALIDATION OF FIT, TUNE AND EVAL
# ==========================================================

expected_files = {
    "fit": "fit_candidates_sampled.parquet",
    "tune": "tune_candidates.parquet",
    "eval": "eval_candidates.parquet"
}

feature_schemas = {}

for split, candidate_file in expected_files.items():

    feature_path = PROCESSED_DIR / f"features_{split}.parquet"
    source_path = PROCESSED_DIR / candidate_file

    saved = pq.ParquetFile(feature_path)
    source = pq.ParquetFile(source_path)

    # Check row counts
    assert saved.metadata.num_rows == source.metadata.num_rows

    # Check that all seven features exist
    assert set(feature_columns).issubset(
        saved.schema_arrow.names
    )

    feature_schemas[split] = {
        col: str(saved.schema_arrow.field(col).type)
        for col in feature_columns
    }

    # Check a sample from the saved dataset
    sample = next(
        saved.iter_batches(
            batch_size=10000,
            columns=["label", "cold_article"] + feature_columns
        )
    ).to_pandas()

    assert sample["label"].isin([0, 1]).all()
    assert sample["cat_match"].between(0, 1).all()
    assert sample["user_article_sim"].between(-1, 1).all()
    assert np.isfinite(sample[feature_columns].to_numpy()).all()

    print(
        f"{split.upper()}: "
        f"{saved.metadata.num_rows:,} rows — PASS"
    )

# Check that all three splits use identical feature types
assert feature_schemas["fit"] == feature_schemas["tune"]
assert feature_schemas["fit"] == feature_schemas["eval"]

print("\nAll three datasets have matching feature schemas!")
print("Final feature engineering validation passed!")


FIT: 12,894,210 rows — PASS
TUNE: 17,400,106 rows — PASS
EVAL: 14,085,557 rows — PASS

All three datasets have matching feature schemas!
Final feature engineering validation passed!
